## Step-by-Step Instructions

### Step 1: Setting Up the Environment

**Objective:** Install required libraries and set up project structure.

**What to do:**
1. Install required packages
2. Set up OpenAI API access
3. Prepare audio file handling

In [1]:
import sys
import importlib.util

print("Python version:", sys.version)
print("Python executable:", sys.executable)

for package in ["openai", "pydub", "dotenv"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if installed else 'Missing'}")

Python version: 3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
Python executable: c:\Users\sabri\miniconda3\envs\bootcamp-env\python.exe
openai: Installed
pydub: Installed
dotenv: Installed


In [2]:
import shutil
import subprocess

ffmpeg_path = shutil.which("ffmpeg")

print("FFmpeg path:", ffmpeg_path)

if ffmpeg_path:
    result = subprocess.run(
        ["ffmpeg", "-version"],
        capture_output=True,
        text=True
    )
    print(result.stdout.splitlines()[0])
else:
    print("FFmpeg is not available on PATH.")

FFmpeg path: C:\Users\sabri\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-9.0.2-full_build\bin\ffmpeg.EXE
ffmpeg version 9.0.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers


In [4]:
#Initialize the OpenAI client

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print("API key configured:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully")

API key configured: True
OpenAI client initialized successfully


In [5]:
#final environment check

from pathlib import Path
import os
from dotenv import load_dotenv
from openai import OpenAI

# Project directories
PROJECT_DIR = Path.cwd()
AUDIO_DIR = PROJECT_DIR / "audio"
OUTPUT_DIR = PROJECT_DIR / "outputs"

AUDIO_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Load API credentials
load_dotenv(PROJECT_DIR / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing")

client = OpenAI()

print("Project:", PROJECT_DIR)
print("Audio directory:", AUDIO_DIR.exists())
print("Output directory:", OUTPUT_DIR.exists())
print("OpenAI client initialized:", isinstance(client, OpenAI))

Project: c:\Users\sabri\OneDrive\Desktop\Ironhack-bootcamp\Week 1\day 5\LAB Whisper STT Implementation\lab-whisper-stt
Audio directory: True
Output directory: True
OpenAI client initialized: True


### Step 2: Downloading Sample Meeting Audio

**Objective:** Get a sample meeting audio file for transcription.

In [6]:
from pydub import AudioSegment

audio_path = AUDIO_DIR / "project_nimbus_meeting.mp3"

audio = AudioSegment.from_file(audio_path)

print(f"Duration: {len(audio) / 1000:.2f} seconds")
print(f"Duration: {len(audio) / 60000:.2f} minutes")
print(f"Channels: {audio.channels}")
print(f"Sample rate: {audio.frame_rate} Hz")
print(f"File size: {audio_path.stat().st_size / (1024**2):.2f} MB")

Duration: 309.20 seconds
Duration: 5.15 minutes
Channels: 1
Sample rate: 22050 Hz
File size: 2.36 MB


In [7]:
from pathlib import Path
from pydub import AudioSegment

audio_path = Path("audio/project_nimbus_meeting.mp3")

print("File exists:", audio_path.exists())

if audio_path.exists():
    audio = AudioSegment.from_file(audio_path)

    print(f"Duration: {len(audio) / 60000:.2f} minutes")
    print(f"Channels: {audio.channels}")
    print(f"Sample rate: {audio.frame_rate} Hz")
    print(f"File size: {audio_path.stat().st_size / (1024**2):.2f} MB")

File exists: True
Duration: 5.15 minutes
Channels: 1
Sample rate: 22050 Hz
File size: 2.36 MB


### Step 3: Basic Transcription (Without Chunking)

**Objective:** Transcribe a short audio file to understand the basic API usage.

**What to do:**
1. Transcribe a short part of the audio file
2. Understand the API response format
3. Extract basic transcription text

In [8]:
from pathlib import Path

audio_path = Path("audio/project_nimbus_meeting.mp3")

with open(audio_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print(transcription.text)

Good morning everyone! This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the RIDIS cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, JIRA number NIM-482, and I expect it to be fixed by Tuesday. Okay, good. Do we need anything from the DevOps team? Yes, we need Tomasz to approve the Terraform changes for the VPC peering. Without that, the pods in the staging namespace cannot reach the legacy database. I can review the Terraform pull request this afternoon. I saw it in GitHub yesterday, but I did not have tim

In [9]:
print("Response type:", type(transcription))
print("Transcription text:", transcription.text[:500])
print("Available fields:", transcription.model_dump().keys())

Response type: <class 'openai.types.audio.transcription.Transcription'>
Transcription text: Good morning everyone! This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the RIDIS cache. When we deploy
Available fields: dict_keys(['text', 'languages', 'logprobs', 'usage'])


### Step 4: Transcription with Prompts (Guided Approach)

**Objective:** Use prompts to guide transcription and improve accuracy.

**What to do:**
1. Create prompts with context (technical terms, names, etc.)
2. Transcribe with prompts
3. Compare results

In [10]:
context_prompt = (
    "Technical vocabulary: Project Nimbus, Kubernetes, "
    "Amazon EKS, PostgreSQL, Redis, Q3 OKRs."
)

with open(audio_path, "rb") as audio_file:
    guided_transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=context_prompt
    )

print(guided_transcription.text[:1000])

Good morning everyone! This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the Redis cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, Jira number NIM-482, and I expect it to be fixed by Tuesday. OK, good. Do we need anything from the DevOps team? Yes, we need Tomasz to approve the Terraform changes for the VPC peering. Without that, the pods in the staging namespace cannot reach the legacy database. I can review the Terraform pull request this afternoon. I saw it in GitHub yesterday, but I did not have time.

In [11]:
print("UNGUIDED:")
print(transcription.text[:1000])

print("\nGUIDED:")
print(guided_transcription.text[:1000])

UNGUIDED:
Good morning everyone! This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the RIDIS cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, JIRA number NIM-482, and I expect it to be fixed by Tuesday. Okay, good. Do we need anything from the DevOps team? Yes, we need Tomasz to approve the Terraform changes for the VPC peering. Without that, the pods in the staging namespace cannot reach the legacy database. I can review the Terraform pull request this afternoon. I saw it in GitHub yesterday, but I did no

In [12]:
from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

(OUTPUT_DIR / "unguided_transcription.txt").write_text(
    transcription.text,
    encoding="utf-8"
)

(OUTPUT_DIR / "guided_transcription.txt").write_text(
    guided_transcription.text,
    encoding="utf-8"
)

print("Both transcriptions saved successfully.")

Both transcriptions saved successfully.


## Step 5 — Unguided Transcription and Comparison

**Objective:** Compare Whisper transcription results with and without contextual prompting.

The unguided transcription was performed in Step 3, and the guided transcription in Step 4. Both used the same audio recording and Whisper model (`whisper-1`).

The comparison focuses on technical terminology recognition and differences in the transcription output.

In [13]:
import difflib

unguided_words = transcription.text.split()
guided_words = guided_transcription.text.split()

matcher = difflib.SequenceMatcher(
    None,
    unguided_words,
    guided_words,
    autojunk=False
)

print("DIFFERENCES BETWEEN TRANSCRIPTIONS\n")

for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag != "equal":
        print(f"Change type: {tag}")
        print("Unguided:", " ".join(unguided_words[i1:i2]))
        print("Guided:  ", " ".join(guided_words[j1:j2]))
        print("-" * 50)

DIFFERENCES BETWEEN TRANSCRIPTIONS

Change type: replace
Unguided: RIDIS
Guided:   Redis
--------------------------------------------------
Change type: replace
Unguided: JIRA
Guided:   Jira
--------------------------------------------------
Change type: replace
Unguided: Okay,
Guided:   OK,
--------------------------------------------------
Change type: replace
Unguided: Xplane Analyze
Guided:   xplane-analyze
--------------------------------------------------
Change type: replace
Unguided: slow
Guided:   sloke
--------------------------------------------------
Change type: replace
Unguided: Datadog
Guided:   metadog
--------------------------------------------------
Change type: replace
Unguided: MPS
Guided:   NPS
--------------------------------------------------
Change type: replace
Unguided: Okay,
Guided:   OK,
--------------------------------------------------
Change type: replace
Unguided: Q3 OKRs.
Guided:   Q3OKRs.
--------------------------------------------------
Change type:

### Step 5 — Comparison Analysis

Both guided and unguided transcription successfully processed the Project Nimbus meeting.

The unguided transcription produced "RIDIS cache", while the guided transcription produced "Redis cache".

I listened to the corresponding section of the original audio and confirmed that the speaker said "Redis". Therefore, the guided transcription correctly recognized this technical term, while the unguided transcription did not.

Other differences, such as "JIRA" versus "Jira" and "Okay" versus "OK", were primarily formatting variations.

**Conclusion:** Contextual prompting improved recognition of the verified technical term "Redis" in this recording. However, this does not establish that the guided transcription is more accurate overall, since the complete transcripts have not been verified against the audio.

### Step 6: Implementing Audio Chunking

**Objective:** Split long audio files into chunks for processing.

**What to do:**
1. Split audio into manageable chunks (e.g., 10-minute segments)
2. Process each chunk separately
3. Combine results with proper timestamps

In [14]:
#create the chunking fnction:

from pydub import AudioSegment
from pathlib import Path

def split_audio(audio_path, chunk_length_ms=60000):
    audio = AudioSegment.from_file(audio_path)
    chunks = []

    for start_ms in range(0, len(audio), chunk_length_ms):
        end_ms = min(start_ms + chunk_length_ms, len(audio))
        chunk = audio[start_ms:end_ms]

        chunks.append({
            "audio": chunk,
            "start_ms": start_ms,
            "end_ms": end_ms
        })

    return chunks

In [15]:
#test the function with a sample audio file

chunks = split_audio(audio_path)

print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks, start=1):
    start_seconds = chunk["start_ms"] / 1000
    end_seconds = chunk["end_ms"] / 1000

    print(
        f"Chunk {i}: "
        f"{start_seconds:.1f}s → {end_seconds:.1f}s"
    )

Total chunks: 6
Chunk 1: 0.0s → 60.0s
Chunk 2: 60.0s → 120.0s
Chunk 3: 120.0s → 180.0s
Chunk 4: 180.0s → 240.0s
Chunk 5: 240.0s → 300.0s
Chunk 6: 300.0s → 309.2s


In [16]:
#export the chunks:

CHUNK_DIR = Path("audio/chunks")
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

for i, chunk in enumerate(chunks, start=1):
    chunk_path = CHUNK_DIR / f"chunk_{i:02d}.mp3"

    chunk["audio"].export(
        chunk_path,
        format="mp3"
    )

    chunk["path"] = chunk_path

    print(f"Saved: {chunk_path}")

Saved: audio\chunks\chunk_01.mp3
Saved: audio\chunks\chunk_02.mp3
Saved: audio\chunks\chunk_03.mp3
Saved: audio\chunks\chunk_04.mp3
Saved: audio\chunks\chunk_05.mp3
Saved: audio\chunks\chunk_06.mp3


In [17]:
#transcribe the chunks using the OpenAI API


chunk_transcriptions = []

for i, chunk in enumerate(chunks, start=1):
    with open(chunk["path"], "rb") as audio_file:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )

    chunk_transcriptions.append({
        "chunk_number": i,
        "start_ms": chunk["start_ms"],
        "end_ms": chunk["end_ms"],
        "text": result.text
    })

    print(f"Chunk {i} transcribed successfully.")

Chunk 1 transcribed successfully.
Chunk 2 transcribed successfully.
Chunk 3 transcribed successfully.
Chunk 4 transcribed successfully.
Chunk 5 transcribed successfully.
Chunk 6 transcribed successfully.


In [18]:
#combine the transcriptions into a single text

combined_transcription = " ".join(
    item["text"] for item in chunk_transcriptions
)

print(combined_transcription)

Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the RIDIS cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, Jira number nimpash482, and I expect it to be fixed by Tuesday. Okay, good. Do we need anything from the DevOps team? Yes, we need Thomas to approve the Terraform changes for the VPC peering. Without that, the pods in the staging namespace cannot reach the legacy database. I can review the Terraform pull request this afternoon. I saw it in GitHub yesterday, but I did not have 

In [19]:
(OUTPUT_DIR / "chunked_transcription.txt").write_text(
    combined_transcription,
    encoding="utf-8"
)

print("Chunked transcription saved.")

Chunked transcription saved.


### Step 7: Transcribing Chunks with Timestamps

**Objective:** Transcribe each chunk and combine with proper timestamps.

**What to do:**
1. Transcribe each chunk
2. Adjust timestamps to account for chunk offset
3. Combine all transcriptions
4. Export with timestamps

**Expected outcome:** You should have a complete transcription with timestamps from chunked audio.

In [ ]:
#transcribe the first chunk with timestamps

with open(chunks[0]["path"], "rb") as audio_file:
    timestamped_result = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="verbose_json",
        timestamp_granularities=["segment"]
    )

print("Available fields:")
print(timestamped_result.model_dump().keys())

print("\nFirst three segments:")

for segment in (timestamped_result.segments or [])[:3]:
    print(
        f"{segment.start:.2f}s → {segment.end:.2f}s: "
        f"{segment.text}"
    )

Available fields:
dict_keys(['duration', 'language', 'text', 'segments', 'usage', 'words', 'task'])

First three segments:
0.00s → 6.00s:  Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today
6.00s → 13.08s:  is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL
13.08s → 20.28s:  latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the


In [21]:
#transcribe all chunks with timestamps

def transcribe_chunks_with_timestamps(chunks, client):
    all_segments = []

    for i, chunk in enumerate(chunks, start=1):
        with open(chunk["path"], "rb") as audio_file:
            result = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                response_format="verbose_json",
                timestamp_granularities=["segment"]
            )

        offset_seconds = chunk["start_ms"] / 1000

        for segment in result.segments:
            all_segments.append({
                "chunk": i,
                "start": round(segment.start + offset_seconds, 2),
                "end": round(segment.end + offset_seconds, 2),
                "text": segment.text.strip()
            })

        print(f"Chunk {i}: timestamps extracted.")

    return all_segments

In [22]:
#run the function to transcribe all chunks with timestamps

timestamped_segments = transcribe_chunks_with_timestamps(
    chunks,
    client
)

print(f"\nTotal segments: {len(timestamped_segments)}")

print("\nFirst 5 segments:")
for segment in timestamped_segments[:5]:
    print(
        f"[{segment['start']:.2f} → {segment['end']:.2f}] "
        f"{segment['text']}"
    )

print("\nLast 3 segments:")
for segment in timestamped_segments[-3:]:
    print(
        f"[{segment['start']:.2f} → {segment['end']:.2f}] "
        f"{segment['text']}"
    )

Chunk 1: timestamps extracted.
Chunk 2: timestamps extracted.
Chunk 3: timestamps extracted.
Chunk 4: timestamps extracted.
Chunk 5: timestamps extracted.
Chunk 6: timestamps extracted.

Total segments: 56

First 5 segments:
[0.00 → 6.00] Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today
[6.00 → 13.08] is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL
[13.08 → 20.28] latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the
[20.28 → 21.28] migration?
[21.28 → 28.28] Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication

Last 3 segments:
[296.92 → 301.84] review and draft the log retention plan with
[300.00 → 305.00] And everyone, please send your greenhouse feedback by Friday.
[305.00 → 308.00] Thanks everyone, see you next week!


In [23]:
overlaps = []

for previous, current in zip(
    timestamped_segments,
    timestamped_segments[1:]
):
    if current["start"] < previous["end"]:
        overlaps.append({
            "previous_chunk": previous["chunk"],
            "current_chunk": current["chunk"],
            "overlap_seconds": round(
                previous["end"] - current["start"], 2
            ),
            "previous_text": previous["text"],
            "current_text": current["text"]
        })

print(f"Detected overlaps: {len(overlaps)}")

for overlap in overlaps:
    print("\n", overlap)

Detected overlaps: 1

 {'previous_chunk': 5, 'current_chunk': 6, 'overlap_seconds': 1.84, 'previous_text': 'review and draft the log retention plan with', 'current_text': 'And everyone, please send your greenhouse feedback by Friday.'}


### Timestamp and Chunk-Boundary Validation

The timestamped transcription produced 56 segments across six audio chunks.

A validation check detected a 1.84-second timestamp overlap between Chunks 5 and 6, near the 300-second boundary.

I listened to the corresponding section of the original audio and identified a missing word. The speaker said "review and draft the log retention plan with Priya," but the chunked transcription omitted "Priya."

This demonstrates that fixed-duration chunking can introduce transcription errors when a chunk boundary interrupts speech.

Potential improvements include splitting audio at natural pauses or using overlapping chunks with a method to remove duplicated content.

### Step 8: Exporting with Timestamps

**Objective:** Export transcriptions in readable formats with timestamps.

This is new. Take your time to research and implement!

**What to do:**
1. Create human-readable format with timestamps
2. Export to text file
3. Export to SRT (subtitle) format
4. Export to JSON


**Expected outcome:** You should have transcriptions exported in multiple formats.

In [29]:
#export the timestamped segments to a text file TXT

from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

txt_path = OUTPUT_DIR / "meeting_transcription.txt"

transcript_text = " ".join(
    segment["text"] for segment in timestamped_segments
)

txt_path.write_text(transcript_text, encoding="utf-8")

print(f"TXT saved: {txt_path}")

TXT saved: outputs\meeting_transcription.txt


In [30]:
def format_srt_time(seconds):
    total_ms = round(seconds * 1000)

    hours, remainder = divmod(total_ms, 3600000)
    minutes, remainder = divmod(remainder, 60000)
    secs, milliseconds = divmod(remainder, 1000)

    return (
        f"{hours:02d}:{minutes:02d}:"
        f"{secs:02d},{milliseconds:03d}"
    )

In [32]:
srt_path = OUTPUT_DIR / "meeting_subtitles.srt"

srt_blocks = []

for i, segment in enumerate(timestamped_segments, start=1):
    start = format_srt_time(segment["start"])
    end = format_srt_time(segment["end"])

    srt_blocks.append(
        f"{i}\n"
        f"{start} --> {end}\n"
        f"{segment['text']}"
    )

srt_path.write_text(
    "\n\n".join(srt_blocks) + "\n",
    encoding="utf-8"
)

print(f"SRT saved: {srt_path}")

SRT saved: outputs\meeting_subtitles.srt


In [33]:
#export the timestamped segments to a JSON file JSON

import json

json_path = OUTPUT_DIR / "meeting_transcription.json"

transcription_data = {
    "source_file": audio_path.name,
    "model": "whisper-1",
    "chunk_count": len(chunks),
    "segment_count": len(timestamped_segments),
    "segments": timestamped_segments
}

json_path.write_text(
    json.dumps(
        transcription_data,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(f"JSON saved: {json_path}")

JSON saved: outputs\meeting_transcription.json


In [34]:
for file_path in [txt_path, srt_path, json_path]:
    print(
        f"{file_path.name}: "
        f"{file_path.exists()} | "
        f"{file_path.stat().st_size} bytes"
    )

print("\nSRT preview:\n")
print(srt_path.read_text(encoding="utf-8")[:500])

meeting_transcription.txt: True | 3762 bytes
meeting_subtitles.srt: True | 5880 bytes
meeting_transcription.json: True | 9252 bytes

SRT preview:

1
00:00:00,000 --> 00:00:06,000
Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today

2
00:00:06,000 --> 00:00:13,080
is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL

3
00:00:13,080 --> 00:00:20,280
latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the

4
00:00:20,280 --> 00:00:21,280
migration?

5
00:00:21,280 --> 00:00:28,280
Sure, thanks Marcus. So the Kubernetes migration is about 70
